# teo_2 — dashboard di training

Serve a rispondere a tre domande, in quest'ordine:

1. **il training sta imparando qualcosa?** → le curve per iterazione (sezione 1)
2. **quello che ha imparato serve a vincere?** → arena fra checkpoint e calibrazione (sezioni 3 e 5)
3. **come gioca, mossa per mossa?** → replay annotato con valutazione e mosse candidate (sezione 4)

La differenza fra 1 e 2 e' il punto centrale del reinforcement learning applicato:
una loss che scende dice solo che la rete sta imitando meglio il proprio bersaglio.
Se il bersaglio e' povero (poche simulazioni MCTS, target di value schiacciati)
la loss scende comunque e l'agente non migliora. Per questo qui non c'e' un solo
grafico di loss: ci sono le metriche che dicono *se il bersaglio vale qualcosa*.

| modulo | cosa fa |
|---|---|
| [`train_metrics.py`](train_metrics.py) | legge `train_log.txt`, disegna le curve, prova a diagnosticare il run |
| [`replay_export.py`](replay_export.py) | gioca partite in locale ed esporta il replay HTML annotato (valutazione + mosse candidate) |
| [`../../view_replays/replay_render.py`](../../view_replays/replay_render.py) | il viewer HTML, condiviso con la dashboard dei replay di Kaggle |

**Prerequisiti:** `torch`, `pandas`, `matplotlib`, il package `cg` alla root del repo,
e le CSV delle carte in `view_replays/card_data/` (gia' presenti nel repo).

In [ ]:
import importlib
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

HERE = Path.cwd()                    # esegui il notebook da agents/teo_2
if str(HERE) not in sys.path:
    sys.path.insert(0, str(HERE))

import train_metrics as tm
import replay_export as rx
for _m in (tm, rx):                  # ricarica le modifiche ai moduli senza riavviare
    importlib.reload(_m)

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 220)
plt.rcParams.update({"figure.dpi": 110, "axes.titlesize": 11, "font.size": 9})

# --- configurazione -------------------------------------------------------
TRAIN_LOG  = HERE / "train_log.txt"          # log di train_selfplay.py
OUT_DIR    = HERE / "out"                    # checkpoint
REPLAY_DIR = HERE / "replays"                # dove finiscono gli HTML annotati
CHECKPOINT = OUT_DIR / "teo2_latest.pth"     # pesi da analizzare
SIMS       = 32                              # simulazioni MCTS per decisione (analisi)
print("checkpoint:", CHECKPOINT, "| esiste:", CHECKPOINT.exists())

## 1. Curve di training

Una riga di `train_log.txt` per iterazione. Cosa guardare, in ordine di importanza:

- **`v_std`** — dispersione dei target di value nel buffer. Se scende sotto ~0.3 il
  training e' rotto a monte: i target sono tutti uguali, quindi la value head impara
  una costante e la MCTS resta senza segnale. E' il bug del TD(λ) descritto nel README,
  ed e' il motivo per cui questa e' la prima curva e non l'ultima.
- **`KL`** — divergenza fra la policy della rete e la distribuzione di visite della MCTS.
  E' *la* metrica di fit della policy: la cross-entropy grezza no, perche' cresce con
  `ln(n_azioni)` e si muove anche quando il fit non cambia. KL che scende = la rete sta
  assorbendo la ricerca; KL piatta per decine di iterazioni = collo di bottiglia
  (di solito troppe poche simulazioni per decisione).
- **`H`** e **`n_act`** — entropia del target e numero medio di azioni legali. Vanno letti
  insieme: H che scende con n_act che cresce e' un buon segno (ricerca piu' decisa in
  posizioni piu' complesse); H che scende solo perche' n_act cala vuol dire partite piu'
  corte e piu' povere.
- **loss value / loss Φ** — le due teste. Sono le curve che *sembrano* piu' importanti e
  che in realta' dicono meno: scendono quasi sempre.

In [ ]:
train_df = tm.parse_train_log(TRAIN_LOG)
print(f"{len(train_df)} iterazioni, {train_df['run'].nunique()} run nel file")
display(train_df.tail(5))

In [ ]:
tm.plot_training(train_df)
plt.show()
tm.plot_progress(train_df)
plt.show()

In [ ]:
# Lettura automatica delle curve: euristiche, non verdetti — servono a sapere
# dove guardare. Le soglie sono quelle documentate nel README di teo_2.
for riga in tm.diagnose(train_df):
    print("•", riga)

## 2. Pretraining sui replay (opzionale)

Se hai lanciato `pretrain_from_replays.py` con l'output rediretto su file, qui vedi
loss e **accuratezza top-1** in validazione: quanto spesso la rete sceglie la stessa
mossa dell'esperto. E' l'unica metrica dove esiste una risposta "giusta" da confrontare,
e per questo e' la piu' facile da leggere di tutto il progetto.

In [ ]:
PRETRAIN_LOG = HERE / "pretrain_log.txt"   # cambia se il tuo log ha un altro nome

if PRETRAIN_LOG.exists():
    pre_df = tm.parse_pretrain_log(PRETRAIN_LOG)
    display(pre_df)
    tm.plot_pretrain(pre_df)
    plt.show()
else:
    print(f"nessun log di pretraining in {PRETRAIN_LOG} — salta questa sezione.\n"
          "Per averlo:  python pretrain_from_replays.py --data data/replays > pretrain_log.txt")

## 3. Checkpoint e arena

Le curve della sezione 1 **non misurano la forza**: dicono quanto bene la rete imita la
propria ricerca. L'unico modo per sapere se l'iterazione 249 gioca meglio della 0 e'
farle scontrare. È la stessa distinzione fra "la loss scende" e "l'Elo sale": in AlphaZero
il gating fra generazioni esiste proprio perche' la prima non implica la seconda.

La cella dell'arena e' **lenta** (ogni partita e' qualche centinaio di decisioni, ognuna con
una MCTS): parti da `GAMES = 4`, `ARENA_SIMS = 16` e alza solo se hai tempo.

In [ ]:
display(tm.checkpoint_table(OUT_DIR))

In [ ]:
# --- ARENA (lenta): due generazioni a confronto ---------------------------
ESEGUI_ARENA = False        # metti True quando vuoi davvero aspettare
CKPT_A = OUT_DIR / "teo2_iter000.pth"
CKPT_B = OUT_DIR / "teo2_latest.pth"
GAMES, ARENA_SIMS = 4, 16

if ESEGUI_ARENA:
    deck = rx.read_deck(HERE / "deck.csv")
    mA, dev = rx.load_model(CKPT_A)
    mB, _   = rx.load_model(CKPT_B)
    pA = rx.Teo2Player(deck, mA, dev, sims=ARENA_SIMS, seed=0, name=CKPT_A.stem)
    pB = rx.Teo2Player(list(deck), mB, dev, sims=ARENA_SIMS, seed=1, name=CKPT_B.stem)
    esito = rx.arena(pA, pB, games=GAMES)
    display(pd.DataFrame([esito["wins"]], index=["vittorie"]))
    print("patte:", esito["draws"], "| non concluse:", esito["non_concluse"])
    print("Nota: con poche partite l'intervallo di confidenza e' enorme. 4 partite non\n"
          "distinguono due agenti simili — servono decine di partite per dire qualcosa.")
else:
    print("arena disattivata (ESEGUI_ARENA = False)")

## 4. Partita annotata

Qui si gioca una partita in locale con il motore `cg` e la si esporta nello stesso viewer
usato per i replay di Kaggle, con tre informazioni in piu' che nei replay scaricati non
esistono:

- **curva di valutazione** sotto la plancia (blu = giocatore 0 avanti, rosso = giocatore 1),
  cliccabile per saltare a quel punto della partita;
- **mosse candidate** nel pannello a sinistra: per ogni decisione le alternative che la MCTS
  ha esplorato, con il valore assegnato (in win %), le visite `N` e il prior `P` della rete;
- **riga di debug** per passo: valore della radice, Φ euristico, quanto e' cambiata la
  valutazione dopo la mossa.

Tre avvertenze per leggerla bene:

1. la valutazione e' quella **della rete di teo_2**, cioe' del modello che stiamo giudicando:
   una discesa dice "la rete pensa di stare peggio", non "la mossa era oggettivamente cattiva";
2. `N` e `P` raccontano due cose diverse. `P` e' l'istinto della rete *prima* di cercare,
   `N` e' dove la ricerca ha effettivamente speso tempo. Quando divergono molto, la ricerca sta
   correggendo la policy — ed e' esattamente il segnale che il training dovrebbe assorbire (KL);
3. le decisioni obbligate (una sola azione legale) non hanno pannello: non c'e' niente da scegliere.

In [ ]:
# Avversario: "self" (stesso checkpoint), "random", oppure il main.py di un altro agente.
AVVERSARIO = HERE.parent / "gharchomp_ex" / "main.py"

model, device = rx.load_model(CHECKPOINT)
deck = rx.read_deck(HERE / "deck.csv")

p0 = rx.Teo2Player(deck, model, device, sims=SIMS, seed=0, name=f"teo_2 ({CHECKPOINT.stem})")
if AVVERSARIO == "random":
    p1 = rx.RandomPlayer(list(deck), seed=1)
elif AVVERSARIO == "self":
    p1 = rx.Teo2Player(list(deck), model, device, sims=SIMS, seed=1, name="teo_2 (specchio)")
else:
    p1 = rx.AgentPlayer(AVVERSARIO)

# L'evaluator usa sempre il modello del lato 0: la curva deve raccontare la partita
# con un solo cervello, altrimenti un salto potrebbe essere solo il cambio di chi stima.
match = rx.play_match(p0, p1, seed=0, evaluator=rx.NetEvaluator(model, device))
dec = match.to_dataframe()
print(match.summary())

In [ ]:
# Salva l'HTML (autonomo, ~3 MB: si apre anche fuori dal notebook) e mostralo qui.
html_path = REPLAY_DIR / rx.default_out_name(match)
rx.export_html(match, html_path)
print("replay ->", html_path)
display(rx.show(match, height=900))

### La stessa partita come curva e come tabella

`equity_persa` e' l'analogo della *centipawn loss* degli scacchi: quanto e' scesa la
valutazione di chi ha appena mosso, fra la sua decisione e la successiva. Contiene pero'
anche la risposta avversaria e la fortuna (pescate, monetine, coin flip degli attacchi):
va usata per **trovare i punti interessanti** della partita, non come voto sulla singola mossa.
Le decisioni obbligate sono escluse (`min_actions=2`).

Il punto finale della curva e' l'**esito reale** (0% o 100%): la distanza fra l'ultima stima
e quel punto dice quanto la rete ha visto arrivare il finale.

E se la curva viene fuori tutta a zig-zag? Non e' un difetto del grafico, e' una diagnosi:
significa che la value head cambia idea di decine di punti fra due micro-decisioni dello
stesso turno, in cui la posizione e' quasi identica. Una value head stabile e' il presupposto
perche' la MCTS possa confrontare rami diversi; se oscilla cosi', il `Q` che leggi nel pannello
delle mosse candidate e' rumoroso quanto lei.

In [ ]:
tm.plot_match_eval(dec, names=(p0.name, p1.name), result=match.result)
plt.show()

print("Decisioni dopo cui teo_2 ha perso piu' valutazione:")
display(match.blunders(player=0, top=8)[
    ["step", "turno", "contesto", "n_azioni", "scelta", "win%_di_chi_muove",
     "equity_persa", "top1", "top1_win%", "top2", "top2_win%"]
])

In [ ]:
# Zoom su una singola decisione: tutte le candidate, non solo le prime tre.
# Prendi uno `step` dalla tabella qui sopra e mettilo qui.
peggiori = match.blunders(player=0, top=1)
STEP = int(peggiori["step"].iloc[0]) if len(peggiori) else 0

d = next(x for x in match.decisions if x["step"] == STEP)
print(f"step {d['step']} · turno {d['turn']} · {d['player_name']} · {d['context']}")
print(d["debug"], "\n")
display(pd.DataFrame(d["options"])[["label", "score", "visits", "prior", "q", "selected"]]
        .rename(columns={"label": "mossa", "score": "win%", "visits": "visite N",
                         "prior": "prior P", "q": "Q", "selected": "scelta"}))

## 5. La value head e' onesta?

Un modello puo' avere una loss di value bassissima ed essere comunque **mal calibrato**:
dire 90% quando la verita' e' 60%. La loss misura l'errore medio, la calibrazione misura
se il numero *significa* quello che dice — ed e' il numero che stai leggendo in ogni curva
di valutazione della sezione 4.

Il test: raccogli tante decisioni, raggruppa per probabilita' predetta, e guarda quante di
quelle partite sono state davvero vinte. Sulla diagonale = calibrato; sotto = troppo ottimista.
Servono diverse partite perche' i bin abbiano abbastanza campioni: parti da 3-4 e alza.

In [ ]:
PARTITE_CALIBRAZIONE = 3    # lenta: ~1-2 min a partita con SIMS=32

matches = [match]
for g in range(1, PARTITE_CALIBRAZIONE):
    matches.append(rx.play_match(p0, p1, seed=10 + g,
                                 evaluator=rx.NetEvaluator(model, device)))

cal = rx.calibration(matches, bins=8)
display(cal)
if not cal.empty:
    tm.plot_calibration(cal)
    plt.show()
    print(f"errore medio di calibrazione: {(cal['errore'] * cal['n']).sum() / cal['n'].sum():+.3f}")
    print("(positivo = la rete e' piu' ottimista di quanto i risultati giustifichino)")

---

### Da riga di comando, senza notebook

```bash
# replay annotato contro un altro agente del repo
python replay_export.py --opponent ../gharchomp_ex/main.py --sims 64

# due generazioni a confronto, tre partite
python replay_export.py --opponent self \
    --checkpoint out/teo2_iter000.pth --opponent-checkpoint out/teo2_latest.pth \
    --games 3 --sims 32
```

### Perche' il prossimo run sia piu' leggibile di questo

- `--eval-games 20` in `train_selfplay.py` aggiunge `wr_vs_random` al log: e' un pavimento
  di sanita' (scopre un agente rotto), non una misura di forza.
- `--log-file train_log.txt` sopravvive alla chiusura del terminale ed e' quello che questa
  dashboard legge.
- Un'arena contro il checkpoint precedente ogni N iterazioni resta la sola misura che
  risponde alla domanda "sta migliorando?".